In [ ]:
from ee_data import *
import geemap
import ee

In [ ]:
import ee
# Trigger the authentication flow.
ee.Authenticate()

In [ ]:
# Initialize the library.
ee.Initialize()

In [ ]:
# https://code.earthengine.google.com/tasks
# Use this page to search and cancel multiple tasks. 
# This page will display tasks that have been submitted until 10 days after they have completed, failed, or cancelled.

ee.data.listOperations()

In [ ]:
 # Load configuration file and set paramters
(
    folder_drive,
    date_start,
    date_end,
    date_median,
    buffer_value,
    bb,
    distric,
    EPSG_code,
) = load_config("./ee_data_config.yml")
month_summer = [6, 8] #calendarRange

# ROI

In [ ]:
# Load ROI from Earth Engine
roi = distric.geometry()
roi = roi.buffer(buffer_value) 

In [ ]:
Map = geemap.Map()
Map.addLayer(roi, None, "ROI")
Map.addLayer(distric, None, "distric")
Map.centerObject(roi, 8)
Map

# NDVI
not needed, check range of values

In [22]:
NDVI = ee.ImageCollection('MODIS/061/MOD13A2').select('NDVI').filter(ee.Filter.date(date_start, date_end)).filterBounds(roi).filter(ee.Filter.calendarRange(month_summer[0], month_summer[1], "month"))

def reduce_NDVI(image):
    def filter(image):
        image = image.select("NDVI").multiply(0.0001)
        return image

    image = image.map(filter)
    NDVI_mean = image.mean().rename("NDVI_mean")

    return NDVI_mean

NDVI = reduce_NDVI(NDVI)

# Export the image to Google Drive
task = ee.batch.Export.image.toDrive(
        image=NDVI,
        description="NDVI",
        region=roi,
        scale=1000,
        maxPixels=10000000000000,
        folder=folder_drive,
        crs="EPSG:" + EPSG_code,
    )
task.start()


# Land Cover

In [11]:
def filter_Corine(image):
    
    # return image.gt(200).clip(roi)
    return image.lt(200).clip(roi)

In [ ]:
Corine = ee.ImageCollection("COPERNICUS/CORINE/V20/100m").select('landcover').filterBounds(roi)

urban = Corine.map(filter_Corine)

# Export the image to Google Drive
task = ee.batch.Export.image.toDrive(
        image=urban.mosaic(),
        description="urban_corine_100_buffer",
        region=roi,
        scale=100,
        maxPixels=10000000000000,
        folder=folder_drive,
        crs="EPSG:" + EPSG_code,
    )
task.start()

Map = geemap.Map()
Map.addLayer(roi, None, "ROI")
Map.addLayer(urban, None, "Corine")
Map.centerObject(roi, 8)
Map

# Elevation Data

In [ ]:
def filter_dem(image):
        
        image.clip(roi)

        return image

In [ ]:
dem_set = ee.ImageCollection("COPERNICUS/DEM/GLO30")
dem  = dem_set.select('DEM')
dem = dem.map(filter_dem)

task = ee.batch.Export.image.toDrive(
        image=dem.mosaic(),
        description="dem_200_buffer",
        region=roi,
        scale=200,
        maxPixels=10000000000000,
        folder=folder_drive,
        crs="EPSG:" + EPSG_code,
            )
task.start()


# MODIS

In [ ]:
# Load MODIS data from Earth Engine
MOD_night = get_MODIS("Night", date_start, date_end, month_summer)
MOD_day = get_MODIS("Day", date_start, date_end, month_summer)

print("MOD images:", MOD_night.size().getInfo())

orig_scale = MOD_day.first().projection().nominalScale().getInfo()
#print("Projection, crs, and crs_transform:", MOD_day.first().projection())
print("Resolution MOD:", orig_scale)

# Filter MODIS data
MOD_night_clean = filter_MODIS(MOD_night, "Night")
MOD_day_clean = filter_MODIS(MOD_day, "Day")

print("MOD images:", MOD_night_clean.size().getInfo())

# Reduce MODIS data
MOD_night_reduced_start_end = reduce_MODIS(MOD_night_clean, roi)
MOD_day_reduced_start_end = reduce_MODIS(MOD_day_clean, roi)

MOD_night_reduced_start_med = reduce_MODIS(
    MOD_night_clean.filterDate(date_start, date_median), roi
)
MOD_day_reduced_start_med = reduce_MODIS(
    MOD_day_clean.filterDate(date_start, date_median), roi
)

MOD_night_reduced_med_end = reduce_MODIS(
    MOD_night_clean.filterDate(date_median, date_end), roi
)
MOD_day_reduced_med_end = reduce_MODIS(
    MOD_day_clean.filterDate(date_median, date_end), roi
)

print("final Image:", MOD_day_reduced_start_end.bandTypes().getInfo())

In [ ]:
Map = geemap.Map()
vis = {'min': -20, 'max': 40, 'palette': ['0000FF','F0FFFF', '8B0000']}
Map.addLayer(MOD_day_reduced_start_med.select("LST_mean"),vis,"LST_mean")
Map.addLayer(roi,None,"ROI")
Map.centerObject(roi, 6)
Map

In [ ]:
# Export MODIS data
export_ee(
    folder_drive,
    MOD_night_reduced_start_end,
    f"MODIS_night_{date_start}_{date_end}",
    1000,
    roi,
    EPSG_code,
)
export_ee(
    folder_drive,
    MOD_day_reduced_start_end,
    f"MODIS_day_{date_start}_{date_end}",
    1000,
    roi,
    EPSG_code,
)
export_ee(
    folder_drive,
    MOD_night_reduced_start_med,
    f"MODIS_night_{date_start}_med",
    1000,
    roi,
    EPSG_code,
)
export_ee(
    folder_drive,
    MOD_day_reduced_start_med,
    f"MODIS_day_{date_start}_med",
    1000,
    roi,
    EPSG_code,
)
export_ee(
    folder_drive,
    MOD_night_reduced_med_end,
    f"MODIS_night_med_{date_end}",
    1000,
    roi,
    EPSG_code,
)
export_ee(
    folder_drive, MOD_day_reduced_med_end,
    f"MODIS_day_med_{date_end}", 1000, roi,  EPSG_code
)

print("Export MODIS Done.")

# Landsat

In [ ]:
# Load Landsat 8 data from Earth Engine
L8 = get_L8(roi, date_start, date_end, month_summer)

orig_scale = L8.first().projection().nominalScale().getInfo()
print("Resolution L8:", orig_scale)

print("L8 images:", L8.size().getInfo())

# Filter Landsat 8 data
L8_clean = filter_L8(L8)

# Reduce Landsat 8 data
L8_reduced_start_end = reduce_L8(L8_clean, roi)
L8_reduced_start_med = reduce_L8(L8_clean.filterDate(date_start, date_median), roi)
L8_reduced_med_end = reduce_L8(L8_clean.filterDate(date_median, date_end), roi)

print("final Image:", L8_reduced_start_med.bandTypes().getInfo())

In [ ]:
Map = geemap.Map()
Map.addLayer(roi,None,"ROI")
vis = {'min': -20, 'max': 40, 'palette': ['0000FF','F0FFFF', '8B0000']}
Map.addLayer( L8_reduced_start_end.select("LST_mean"),vis,"LST_mean")
Map.centerObject(roi, 6)
Map

In [ ]:
scale_m = 100 # 30
param_band = 'LST_p90' #"LST_mean"

# Export Landsat 8 data
export_ee(
    folder_drive,
    L8_reduced_start_end,#.select(param_band),
    f"L8_{date_start}_{date_end}",
    scale_m ,
    roi,
    EPSG_code,
)
export_ee(
    folder_drive,
    L8_reduced_start_med,#.select(param_band ),
    f"L8_{date_start}_med",
    scale_m,
    roi,
    EPSG_code,
)
export_ee(
    folder_drive,
    L8_reduced_med_end,#.select(param_band),
    f"L8_med_{date_end}",
    scale_m,
    roi,
    EPSG_code,
)

print("Export L8 Done.")